# Prosperity Trading Dashboard

This notebook loads your Prosperity backtester log and plots:
- The **best bid** and **best ask** prices over time (the market)
- Your **own trades** (triangles on the chart)

The log file is a single JSON file. Inside it there are two things we care about:
1. `activitiesLog` — a CSV string with the order book snapshots at every timestamp
2. `tradeHistory` — a list of every trade that happened, including yours

In [1]:
# Cell 1 — Imports
# pandas: for working with tabular data (DataFrames)
# plotly: for making interactive charts
# json and io: built-in Python tools to parse JSON and read strings as files

import json
import io
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Cell 2 — Load the log file
#
# The log file is one big JSON object. We open it and parse two parts:
#   - prices_df : order book snapshots (one row per product per timestamp)
#   - trades_df : every trade that happened (market trades + your trades)

LOG_PATH = "../data/26822.log"

with open(LOG_PATH) as f:
    log = json.load(f)

# activitiesLog is a CSV stored as a string inside the JSON.
# io.StringIO lets pandas read it as if it were a file.
prices_df = pd.read_csv(io.StringIO(log["activitiesLog"]), sep=";")

# tradeHistory is already a list of dicts — easy to turn into a DataFrame.
trades_df = pd.DataFrame(log["tradeHistory"])

print("Order book shape:", prices_df.shape)
print("Columns:", list(prices_df.columns))
print()
print("Trade history shape:", trades_df.shape)
print(trades_df.head())

Order book shape: (4000, 17)
Columns: ['day', 'timestamp', 'product', 'bid_price_1', 'bid_volume_1', 'bid_price_2', 'bid_volume_2', 'bid_price_3', 'bid_volume_3', 'ask_price_1', 'ask_volume_1', 'ask_price_2', 'ask_volume_2', 'ask_price_3', 'ask_volume_3', 'mid_price', 'profit_and_loss']

Trade history shape: (105, 7)
   timestamp buyer      seller    symbol currency   price  quantity
0          0        SUBMISSION  EMERALDS   XIRECS  9992.0        15
1        100        SUBMISSION  EMERALDS   XIRECS  9992.0        13
2        200        SUBMISSION  EMERALDS   XIRECS  9992.0        13
3        300        SUBMISSION  EMERALDS   XIRECS  9992.0        11
4        400        SUBMISSION  EMERALDS   XIRECS  9992.0        13


In [3]:
# Cell 3 — Filter to one product and extract what we need
#
# Change PRODUCT to whichever symbol you want to inspect.
# Available products: EMERALDS, TOMATOES

PRODUCT = "EMERALDS"

# --- Order book: keep only rows for this product ---
# Each row has: timestamp, bid_price_1 (best bid), ask_price_1 (best ask)
ob = prices_df[prices_df["product"] == PRODUCT][["timestamp", "bid_price_1", "ask_price_1"]].copy()

print("Order book sample:")
print(ob.head())
print()

# --- My trades: rows where SUBMISSION is buyer or seller ---
# buyer == 'SUBMISSION'  → we bought
# seller == 'SUBMISSION' → we sold
my_trades = trades_df[
    (trades_df["symbol"] == PRODUCT) &
    ((trades_df["buyer"] == "SUBMISSION") | (trades_df["seller"] == "SUBMISSION"))
].copy()

my_buys  = my_trades[my_trades["buyer"]  == "SUBMISSION"]
my_sells = my_trades[my_trades["seller"] == "SUBMISSION"]

print(f"My buys:  {len(my_buys)} trades")
print(f"My sells: {len(my_sells)} trades")
print(my_trades[["timestamp", "buyer", "seller", "price", "quantity"]])

Order book sample:
   timestamp  bid_price_1  ask_price_1
1          0         9992        10008
2        100         9992        10008
4        200         9992        10008
7        300         9992        10008
9        400         9992        10008

My buys:  0 trades
My sells: 6 trades
   timestamp buyer      seller   price  quantity
0          0        SUBMISSION  9992.0        15
1        100        SUBMISSION  9992.0        13
2        200        SUBMISSION  9992.0        13
3        300        SUBMISSION  9992.0        11
4        400        SUBMISSION  9992.0        13
5        500        SUBMISSION  9992.0        12


In [4]:
# Cell 4 — Plot
#
# We draw three things on the same chart:
#   1. Best bid price over time  (blue line)
#   2. Best ask price over time  (red line)
#   3. My buy trades             (green triangle pointing UP)
#   4. My sell trades            (orange triangle pointing DOWN)

fig = go.Figure()

# --- 1. Best bid line ---
fig.add_trace(go.Scatter(
    x=ob["timestamp"],
    y=ob["bid_price_1"],
    mode="lines",
    name="Best Bid",
    line=dict(color="blue"),
))

# --- 2. Best ask line ---
fig.add_trace(go.Scatter(
    x=ob["timestamp"],
    y=ob["ask_price_1"],
    mode="lines",
    name="Best Ask",
    line=dict(color="red"),
))

# --- 3. My buy trades ---
# triangle-up = I bought at this price at this time
if not my_buys.empty:
    fig.add_trace(go.Scatter(
        x=my_buys["timestamp"],
        y=my_buys["price"],
        mode="markers",
        name="My Buys",
        marker=dict(symbol="triangle-up", size=14, color="lime"),
    ))

# --- 4. My sell trades ---
# triangle-down = I sold at this price at this time
if not my_sells.empty:
    fig.add_trace(go.Scatter(
        x=my_sells["timestamp"],
        y=my_sells["price"],
        mode="markers",
        name="My Sells",
        marker=dict(symbol="triangle-down", size=14, color="orange"),
    ))

# --- Layout ---
fig.update_layout(
    title=f"{PRODUCT} — Bid / Ask + My Trades",
    xaxis_title="Timestamp",
    yaxis_title="Price",
    hovermode="x unified",   # hover shows all traces at once
)

fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed